In [1]:
# Generate search result urls

In [2]:
import time, difflib, re, pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup
from urllib.parse import quote_plus, urlparse
from tqdm import tqdm
from datetime import datetime
import socket
import re

In [3]:
class joongangNewsCrawler:
    def __init__(self, df_with_urls):  # url, keyword, industry 포함된 df
        self.df = df_with_urls.copy()

    def crawl(self):
        driver = webdriver.Chrome()
        result = []

        for i, row in self.df.iterrows():
            url = row['url']
            industry = row['industry']
            keyword = row['keyword']

            try:
                driver.get(url)
                time.sleep(2)
                soup = BeautifulSoup(driver.page_source, 'lxml')

                title = soup.select_one('h1.headline')
                content = soup.select('div.article_body.fs3 > p')
                date = soup.select_one('time[itemprop="datePublished"]')


                title_text = title.get_text(strip=True) if title else ''
                content_text = '\n'.join(tag.get_text(strip=True) for tag in content)
                date_text = date.get_text(strip=True) if date else ''


                
                result.append({
                    'industry': industry,
                    'keyword': keyword,
                    'date': date_text,
                    'title': title_text,
                    'content': content_text,
                    'url': url
                })

            except Exception as e:
                print(f"❌ {url} 실패: {e}")
                continue

        driver.quit()
        return pd.DataFrame(result)


In [4]:
class ThelecNewsCrawler:
    def __init__(self, df_with_urls):
        self.df = df_with_urls.copy()

    def crawl(self):
        from selenium import webdriver
        from bs4 import BeautifulSoup
        import time
        import re

        driver = webdriver.Chrome()
        result = []

        for i, row in self.df.iterrows():
            url = row['url']
            industry = row['industry']
            keyword = row['keyword']

            try:
                driver.get(url)
                time.sleep(2)
                soup = BeautifulSoup(driver.page_source, 'lxml')

                # ✅ 제목
                title_tag = soup.select_one('div.article-head-title')
                title_text = title_tag.get_text(strip=True) if title_tag else ''

                # ✅ 본문
                content_tags = soup.select('div#article-view-content-div p')
                content_text = '\n'.join([
                    p.get_text(strip=True)
                    for p in content_tags
                    if len(p.get_text(strip=True)) > 10
                ])


                # ✅ 날짜
                date_text = ''
                li_tags = soup.select('div.info-text li')
                for li in li_tags:
                    full_text = li.get_text(" ", strip=True)
                    match = re.search(r'\d{4}[.]\d{2}[.]\d{2}\s\d{2}:\d{2}', full_text)
                    if match:
                        date_text = match.group(0)
                        break

                # ✅ 저장
                result.append({
                    'industry': industry,
                    'keyword': keyword,
                    'date': date_text,
                    'title': title_text,
                    'content': content_text,
                    'url': url
                })

            except Exception as e:
                print(f"❌ {url} 실패: {e}")
                continue

        driver.quit()
        return pd.DataFrame(result)


In [5]:
class ElectimesNewsCrawler:
    def __init__(self, df_with_urls):
        self.df = df_with_urls.copy()

    def crawl(self):
        from selenium import webdriver
        from bs4 import BeautifulSoup
        import time
        import re

        driver = webdriver.Chrome()
        result = []

        for i, row in self.df.iterrows():
            url = row['url']
            industry = row['industry']
            keyword = row['keyword']

            try:
                driver.get(url)
                time.sleep(2)
                soup = BeautifulSoup(driver.page_source, 'lxml')

                # ✅ 제목
                title_tag = soup.select_one('h3.heading')

                title_text = title_tag.get_text(strip=True) if title_tag else ''

                # ✅ 본문
                container = soup.select_one('article#article-view-content-div.article-veiw-body')
                paragraphs = container.select('p') if container else []
                content_text = '\n'.join(p.get_text(strip=True) for p in paragraphs)

                # ✅ 날짜 (정규식으로도 보완 가능)
                date_text = ''
                date_candidates = soup.select('ul.info li')
                
                for li in date_candidates:
                    if '입력' in li.text:
                        date_text = li.text.strip().replace('입력', '').strip()
                        break

                result.append({
                    'industry': industry,
                    'keyword': keyword,
                    'date': date_text,
                    'title': title_text,
                    'content': content_text,
                    'url': url
                })

            except Exception as e:
                print(f"❌ {url} 실패: {e}")
                continue

        driver.quit()
        return pd.DataFrame(result)


In [6]:
class YnaNewsCrawler:
    def __init__(self, df_with_urls):
        self.df = df_with_urls.copy()

    def crawl(self):
        from selenium import webdriver
        from bs4 import BeautifulSoup
        import time
        import re

        driver = webdriver.Chrome()
        result = []

        for i, row in self.df.iterrows():
            url = row['url']
            industry = row['industry']
            keyword = row['keyword']

            try:
                driver.get(url)
                time.sleep(2)
                soup = BeautifulSoup(driver.page_source, 'lxml')

                # ✅ 제목
                title_tag = soup.select_one('h1.tit') or soup.select_one('h1.tit-article')
                title_text = title_tag.get_text(strip=True) if title_tag else ''

                # ✅ 본문
                container = soup.select_one('div.story-news.article')
                paragraphs = container.select('p') if container else []
                content_text = '\n'.join(p.get_text(strip=True) for p in paragraphs)

                # ✅ 날짜
                date_text = ''
                meta_tag = soup.find('meta', {'property': 'article:published_time'})
                if meta_tag:
                    date_text = meta_tag.get('content', '').replace('T', ' ').split('+')[0]
                else:
                    date_tag = soup.select_one('p.info-date') or soup.select_one('span.update-time')
                    if date_tag:
                        date_text = date_tag.get_text(strip=True)

                result.append({
                    'industry': industry,
                    'keyword': keyword,
                    'date': date_text,
                    'title': title_text,
                    'content': content_text,
                    'url': url
                })

            except Exception as e:
                print(f"❌ {url} 실패: {e}")
                continue

        driver.quit()
        return pd.DataFrame(result)


In [7]:
class seoulfnNewsCrawler:
    def __init__(self, df_with_urls):  # df에는 최소한 'url', 'keyword' 포함되어 있어야 함
        self.df = df_with_urls.copy()

    def crawl(self):
        driver = webdriver.Chrome()
        result = []

        for i, row in self.df.iterrows():
            url = row['url']
            industry = row['industry']
            keyword = row['keyword']


            try:
                driver.get(url)
                time.sleep(2)
                soup = BeautifulSoup(driver.page_source, 'lxml')

                title_tag = soup.select_one('div.article-head-title')
                content_tags = soup.select('div#article-view-content-div p')
                          
                title_text = title_tag.get_text(strip=True) if title_tag else ''
                content_text = '\n'.join(tag.get_text(strip=True) for tag in content_tags)

                # 날짜 추출 및 datetime 변환
                date_text = ''
                li_tags = soup.select('div.info-text li')
                for li in li_tags:
                    full_text = li.get_text(" ", strip=True)
                    match = re.search(r'\d{4}[.]\d{2}[.]\d{2}\s\d{2}:\d{2}', full_text)
                    if match:
                        date_text = match.group(0)
                        break

                result.append({
                    'industry': industry,
                    'keyword': keyword,
                    'date': date_text,
                    'title': title_text,
                    'content': content_text,
                    'url': url
                })

            except Exception as e:
                print(f"❌ {url} 실패: {e}")
                continue

        driver.quit()
        return pd.DataFrame(result)


In [8]:
class ChosunBizNewsCrawler:
    def __init__(self, df_with_urls):  # url, keyword, industry 포함된 df
        self.df = df_with_urls.copy()

    def crawl(self):
        from selenium import webdriver
        from bs4 import BeautifulSoup
        import time

        driver = webdriver.Chrome()
        result = []

        for i, row in self.df.iterrows():
            url = row['url']
            industry = row['industry']
            keyword = row['keyword']

            try:
                driver.get(url)
                time.sleep(2)
                soup = BeautifulSoup(driver.page_source, 'lxml')

                # ✅ 제목
                title_tag = soup.select_one('h1') or soup.select_one('h1.headline')
                title_text = title_tag.get_text(strip=True) if title_tag else ''

                # ✅ 본문 (조선비즈 구조: section.article-body > p)
                content_text = ''
                container = soup.select_one('section.article-body')
                if container:
                    paragraphs = container.select('p')
                    lines = [p.get_text(" ", strip=True) for p in paragraphs if len(p.get_text(strip=True)) > 5]
                    content_text = "\n".join(lines)

                # ✅ 날짜 (메타 태그 또는 일반 텍스트)
                date_tag = soup.select_one('span.inputDate')
                date_text = ''
                
                if date_tag:
                    date_text = date_tag.text.strip().replace('입력', '').strip()

                # ✅ 결과 저장
                result.append({
                    'industry': industry,
                    'keyword': keyword,
                    'date': date_text,
                    'title': title_text,
                    'content': content_text,
                    'url': url
                })

            except Exception as e:
                print(f"❌ {url} 실패: {e}")
                continue

        driver.quit()
        return pd.DataFrame(result)


In [9]:
class GEnewsCrawler:
    def __init__(self, df_with_urls):
        self.df = df_with_urls.copy()

    def crawl(self):
        driver = webdriver.Chrome()
        result = []

        for _, row in self.df.iterrows():
            url = row['url']
            keyword = row.get('keyword', '')

            try:
                driver.get(url)
                time.sleep(2)
                soup = BeautifulSoup(driver.page_source, 'lxml')

                # 제목
                title_tag = soup.select_one('div.vtop h1')
                title_text = title_tag.get_text(strip=True) if title_tag else ''

                # 날짜
                date_tag = soup.select_one('p.r3 span')
                date_text = date_tag.get_text(strip=True) if date_tag else ''

                # 본문
                content_tag = soup.select_one('div.vtxt.detailCont')
                content_text = content_tag.get_text("\n", strip=True) if content_tag else ''

                # 꼬리말 제거
                remove_patterns = [
                    r'[가-힣]{2,}\s기자.*',                # 기자 이름
                    r'ⓒ.∗?ⓒ.*?',                          # 저작권 문구
                    r'#\S+',                              # 해시태그 (#GS건설 등)
                    r'\n{2,}',                            # 연속 줄바꿈 정리
                ]
                for pattern in remove_patterns:
                    content_text = re.sub(pattern, '', content_text, flags=re.MULTILINE)

                result.append({
                    'industry': row.get('industry', ''),
                    'keyword': keyword,
                    'date': date_text,
                    'title': title_text,
                    'content': content_text,
                    'url': url
                })

            except Exception as e:
                print(f"❌ {url} 실패: {e}")
                continue

        driver.quit()
        return pd.DataFrame(result)

In [10]:
class joseNewsCrawler:
    def __init__(self, df_with_urls):
        self.df = df_with_urls.copy()

    def crawl(self):
        driver = webdriver.Chrome()
        result = []

        for i, row in self.df.iterrows():
            url = row['url']
            keyword = row.get('keyword', '')

            try:
                driver.get(url)
                time.sleep(2)
                soup = BeautifulSoup(driver.page_source, 'lxml')

                # 제목
                title_tag = soup.select_one('div.news_title > h1')
                title_text = title_tag.get_text(strip=True) if title_tag else ''

                # 날짜 (두 방식 모두 시도)
                date_text = ''
                author_tag = soup.select_one('div.news_author')
                if author_tag:
                    full_text = author_tag.get_text(" ", strip=True)
                else:
                    full_text = soup.get_text(" ", strip=True)  # 페이지 전체 텍스트 백업

                # 날짜 정규식 추출
                match = re.search(r'\d{4}[.]\d{1,2}[.]\d{1,2}\s+\d{1,2}:\d{2}', full_text)
                if match:
                    date_text = match.group(0)


                # 본문 (광고 제거, <br> 포함 줄바꿈 반영)
                content_text = ''
                content_root = soup.select_one('div[itemprop="articleBody"]')
                if content_root:
                    content_text = content_root.get_text("\n", strip=True)

                    # 불필요한 꼬리말 제거
                    content_text = re.sub(r'저작권자.∗?무단전재및재배포금지저작권자.*?무단전재 및 재배포 금지', '', content_text)

                result.append({
                    'industry': row.get('industry', ''),
                    'keyword': keyword,
                    'date': date_text,
                    'title': title_text,
                    'content': content_text,
                    'url': url
                })

            except Exception as e:
                print(f"❌ {url} 실패: {e}")
                continue

        driver.quit()
        return pd.DataFrame(result)


In [11]:
class industryNewsCrawler:
    def __init__(self, df_with_urls):  # df에는 최소한 'url', 'keyword' 포함되어 있어야 함
        self.df = df_with_urls.copy()

    def crawl(self):
        driver = webdriver.Chrome()
        result = []

        for i, row in self.df.iterrows():
            url = row['url']
            keyword = row.get('keyword', '')  # keyword 컬럼이 없으면 빈 문자열

            try:
                driver.get(url)
                time.sleep(2)
                soup = BeautifulSoup(driver.page_source, 'lxml')

                #제목
                title_tag = soup.select_one('div.article-head-title')
                title_text = title_tag.get_text(strip=True) if title_tag else ''

                #본문
                content_tag = soup.select_one('#article-view-content-div')
                # 본문 추출 후 후처리
                content_text = content_tag.get_text("\n", strip=True) if content_tag else ''
                
                remove_patterns = [
                    r'\n?저작권자\s+©\s+인더스트리뉴스\s+무단전재\s+및\s+재배포\s+금지\n?',  # 저작권
                    r'\n?.{2,10}\s기자\n?',  # 기자 이름 (예: 정형우 기자)
                    r'\n?다른기사\s보기\n?'  # 다른기사 보기
                ]
                
                for pattern in remove_patterns:
                    content_text = re.sub(pattern, '', content_text)

                #날짜
                date_text = ''
                li_tags = soup.select('div.info-text li')

                for li in li_tags:
                    text = li.get_text(" ", strip=True)
                    if "승인" in text:
                        match = re.search(r'\d{4}[.]\d{2}[.]\d{2}\s\d{2}:\d{2}', text)
                        if match:
                            date_text = match.group(0)
                            break

                result.append({
                    'industry': row.get('industry', ''), 
                    'keyword': keyword,
                    'date': date_text,
                    'title': title_text,
                    'content': content_text,
                    'url': url
                })

            except Exception as e:
                print(f"❌ {url} 실패: {e}")
                continue

        driver.quit()
        return pd.DataFrame(result)


In [12]:
class SegyeNewsCrawler:
    def __init__(self, df_with_urls):
        self.df = df_with_urls.copy()

    def crawl(self):
        driver = webdriver.Chrome()
        result = []

        for _, row in self.df.iterrows():
            url = row['url']
            keyword = row.get('keyword', '')

            try:
                driver.get(url)
                time.sleep(2)
                soup = BeautifulSoup(driver.page_source, 'lxml')

                # 제목
                title_tag = soup.select_one('section#contTitle h3#title_sns')
                title_text = title_tag.get_text(strip=True) if title_tag else ''

                # 날짜
                date_tag = soup.select_one('section#contTitle p.viewInfo')
                date_text = ''
                if date_tag:
                    match = re.search(r'입력\s*:\s*([\d\-. :]+)', date_tag.get_text())
                    if match:
                        date_text = match.group(1)

                # 본문
                content_tag = soup.select_one('div#article_txt article.viewBox2')
                content_text = content_tag.get_text("\n", strip=True) if content_tag else ''

                # 꼬리말 제거
                remove_patterns = [
                    r'ⓒ 세계일보 & Segye\.com.*?ⓒ 세계일보 & Segye\.com.*?',
                    r'\s*[가-힣]+\s기자.*',
                ]
                for pattern in remove_patterns:
                    content_text = re.sub(pattern, '', content_text, flags=re.MULTILINE)

                result.append({
                    'industry': row.get('industry', ''),
                    'keyword': keyword,
                    'date': date_text,
                    'title': title_text,
                    'content': content_text,
                    'url': url
                })

            except Exception as e:
                print(f"❌ {url} 실패: {e}")
                continue

        driver.quit()
        return pd.DataFrame(result)


In [13]:
class sbsNewsCrawler:
    def __init__(self, df_with_urls):  # df에는 최소한 'url', 'keyword' 포함되어 있어야 함
        self.df = df_with_urls.copy()

    def crawl(self):
        driver = webdriver.Chrome()
        result = []

        for i, row in self.df.iterrows():
            url = row['url']
            keyword = row.get('keyword', '')  # keyword 컬럼이 없으면 빈 문자열

            try:
                driver.get(url)
                time.sleep(2)
                soup = BeautifulSoup(driver.page_source, 'lxml')

                title_tag = soup.select_one('h1.article_main_tit')
                content_tags = soup.select_one('div.text_area[itemprop="articleBody"]')
                date_tag = soup.select_one('div.date_area span')


                title_text = title_tag.get_text(strip=True) if title_tag else ''
                content_text = '\n'.join(tag.get_text(strip=True) for tag in content_tags)
                date_text = date_tag.get_text(strip=True) if date_tag else ''

                result.append({
                    'industry': row.get('industry', ''),  # ✅ 나중에 industry 붙일 예정
                    'keyword': keyword,
                    'date': date_text,
                    'title': title_text,
                    'content': content_text,
                    'url': url
                })

            except Exception as e:
                print(f"❌ {url} 실패: {e}")
                continue

        driver.quit()
        return pd.DataFrame(result)


In [14]:
class KyeonggiNewsCrawler:
    def __init__(self, df_with_urls):  # df에는 최소한 'url', 'keyword' 포함되어 있어야 함
        self.df = df_with_urls.copy()

    def crawl(self):
        driver = webdriver.Chrome()
        result = []

        for i, row in self.df.iterrows():
            url = row['url']
            keyword = row.get('keyword', '')  # keyword 컬럼이 없으면 빈 문자열

            try:
                driver.get(url)
                time.sleep(2)
                soup = BeautifulSoup(driver.page_source, 'lxml')

                title_tag = soup.select_one('h1.article_tit')
                content_tags = soup.select('div.article_cont_wrap p')
                date_tag = soup.select_one('div.article_date span:nth-of-type(2)')

                title_text = title_tag.get_text(strip=True) if title_tag else ''
                content_text = '\n'.join(tag.get_text(strip=True) for tag in content_tags)
                date_text = date_tag.get_text(strip=True) if date_tag else ''

                result.append({
                    'industry': row.get('industry', ''),
                    'keyword': keyword,
                    'date': date_text,
                    'title': title_text,
                    'content': content_text,
                    'url': url
                })

            except Exception as e:
                print(f"❌ {url} 실패: {e}")
                continue

        driver.quit()
        return pd.DataFrame(result)


In [ ]:
# Dedupe

In [1]:
import pandas as pd

df = pd.read_csv("./google/v4/first_half_articles.csv")  # or whatever your path is

df = df.drop_duplicates(subset=["url"], keep="first")

# 3) (Optionally) write back out
df.to_csv("./google/v4/first_half_articles.csv", index=False, encoding="utf-8-sig")

In [ ]:
# Main Crawl

In [15]:
import pandas as pd

# load the master CSV once
df = pd.read_csv("./google/v4/first_half_articles.csv")

# 1) Joongang
df_joongang = df[df['url'].str.contains("joongang.co.kr", case=False, na=False)].copy()
crawler_joongang = joongangNewsCrawler(df_joongang)
df_cleaned_joongang = crawler_joongang.crawl()
print("Joongang:", len(df_cleaned_joongang), "articles")
#df_cleaned_joongang.to_csv("./google/v3/v3_joongang_20210_jy.csv", index=False, encoding='utf-8-sig')

# 2) Thelec
df_thelec = df[df['url'].str.contains("thelec.kr", case=False, na=False)].copy()
crawler_thelec = ThelecNewsCrawler(df_thelec)
df_cleaned_thelec = crawler_thelec.crawl()
print("Thelec:", len(df_cleaned_thelec), "articles")
#df_cleaned_thelec.to_csv("./google/v3/v3_thelec_2020_jy.csv", index=False, encoding='utf-8-sig')

# 3) Electimes
df_electimes = df[df['url'].str.contains("electimes.com", case=False, na=False)].copy()
crawler_electimes = ElectimesNewsCrawler(df_electimes)
df_cleaned_electimes = crawler_electimes.crawl()
print("Electimes:", len(df_cleaned_electimes), "articles")
df_cleaned_electimes.to_csv("./google/v3/v3_electimes_2020_jy.csv", index=False, encoding='utf-8-sig')

# 4) YNA
df_yna = df[df['url'].str.contains("yna.co.kr", case=False, na=False)].copy()
crawler_yna = YnaNewsCrawler(df_yna)
df_cleaned_yna = crawler_yna.crawl()
print("YNA:", len(df_cleaned_yna), "articles")
#df_cleaned_yna.to_csv("./google/v3/v3_yna_2020_jy.csv", index=False, encoding='utf-8-sig')

# 5) SeoulFN
df_seoulfn = df[df['url'].str.contains("seoulfn.com", case=False, na=False)].copy()
crawler_seoulfn = seoulfnNewsCrawler(df_seoulfn)
df_cleaned_seoulfn = crawler_seoulfn.crawl()
print("SeoulFN:", len(df_cleaned_seoulfn), "articles")
#df_cleaned_seoulfn.to_csv("./google/v3/v3_seoulfn_2020_jy.csv", index=False, encoding='utf-8-sig')

# 6) Chosun Biz
df_chosun = df[df['url'].str.contains("biz.chosun.com", case=False, na=False)].copy()
crawler_chosun = ChosunBizNewsCrawler(df_chosun)
df_cleaned_chosun = crawler_chosun.crawl()
print("ChosunBiz:", len(df_cleaned_chosun), "articles")
#df_cleaned_chosun.to_csv("./google/v3/v3_chosun_2020_jy.csv", index=False, encoding='utf-8-sig')

# 7) G-Enews
df_genews = df[df['url'].str.contains("g-enews.com", case=False, na=False)].copy()
crawler_genews = GEnewsCrawler(df_genews)
df_cleaned_genews = crawler_genews.crawl()
print("G-Enews:", len(df_cleaned_genews), "articles")
#df_cleaned_genews.to_csv("./google/v3/v3_genews_2020_jy.csv", index=False, encoding='utf-8-sig')

# 8) Joseilbo (mobile)
df_jose = df[df['url'].str.contains("m.joseilbo.com", case=False, na=False)].copy()
crawler_jose = joseNewsCrawler(df_jose)
df_cleaned_jose = crawler_jose.crawl()
print("Joseilbo:", len(df_cleaned_jose), "articles")
#df_cleaned_jose.to_csv("./google/v3/v3_jose_2020_jy.csv", index=False, encoding='utf-8-sig')

# 9) IndustryNews
df_industry = df[df['url'].str.contains("industrynews.co.kr", case=False, na=False)].copy()
crawler_industry = industryNewsCrawler(df_industry)
df_cleaned_industry = crawler_industry.crawl()
print("IndustryNews:", len(df_cleaned_industry), "articles")
#df_cleaned_industry.to_csv("./google/v3/v3_industry_2020_jy.csv", index=False, encoding='utf-8-sig')

# 10) Segye
df_segye = df[df['url'].str.contains("segye.com", case=False, na=False)].copy()
crawler_segye = SegyeNewsCrawler(df_segye)
df_cleaned_segye = crawler_segye.crawl()
print("Segye:", len(df_cleaned_segye), "articles")
#df_cleaned_segye.to_csv("./google/v3/v3_segye_2020_jy.csv", index=False, encoding='utf-8-sig')

# 11) SBS
df_sbs = df[df['url'].str.contains("news.sbs.co.kr", case=False, na=False)].copy()
crawler_sbs = sbsNewsCrawler(df_sbs)
df_cleaned_sbs = crawler_sbs.crawl()
print("SBS:", len(df_cleaned_sbs), "articles")
#df_cleaned_sbs.to_csv("./google/v3/v3_sbs_2020_jy.csv", index=False, encoding='utf-8-sig')

# 12) Kyeonggi
df_kyeonggi = df[df['url'].str.contains("kyeonggi.com", case=False, na=False)].copy()
crawler_kyeonggi = KyeonggiNewsCrawler(df_kyeonggi)
df_cleaned_kyeonggi = crawler_kyeonggi.crawl()
print("Kyeonggi:", len(df_cleaned_kyeonggi), "articles")
#df_cleaned_kyeonggi.to_csv("./google/v3/v3_kyeonggi_2020_jy.csv", index=False, encoding='utf-8-sig')


Joongang: 50 articles
Thelec: 29 articles
Electimes: 5 articles
YNA: 366 articles
SeoulFN: 47 articles
ChosunBiz: 434 articles
G-Enews: 154 articles
Joseilbo: 1 articles
IndustryNews: 29 articles
Segye: 45 articles
❌ https://news.sbs.co.kr/news/endPage.do?news_id=N1008127667 실패: 'NoneType' object is not iterable
❌ https://news.sbs.co.kr/amp/news.amp?news_id=N1008125380 실패: 'NoneType' object is not iterable
SBS: 28 articles
❌ https://www.kyeonggi.com/article/20250613580138 실패: Alert Text: 삭제된 기사입니다.
Message: unexpected alert open: {Alert text : 삭제된 기사입니다.}
  (Session info: chrome=137.0.7151.122)
Stacktrace:
	GetHandleVerifier [0x0x7ff61489cda5+78885]
	GetHandleVerifier [0x0x7ff61489ce00+78976]
	(No symbol) [0x0x7ff614659bca]
	(No symbol) [0x0x7ff614701de0]
	(No symbol) [0x0x7ff6146d8963]
	(No symbol) [0x0x7ff6146a16b1]
	(No symbol) [0x0x7ff6146a2443]
	GetHandleVerifier [0x0x7ff614b74eed+3061101]
	GetHandleVerifier [0x0x7ff614b6f33d+3037629]
	GetHandleVerifier [0x0x7ff614b8e592+3165202]


In [17]:
df_all = pd.concat([df_cleaned_joongang, df_cleaned_thelec], ignore_index=True)
df_all = pd.concat([df_all, df_cleaned_electimes], ignore_index=True)
df_all = pd.concat([df_all, df_cleaned_yna], ignore_index=True)
df_all = pd.concat([df_all, df_cleaned_seoulfn], ignore_index=True)
df_all = pd.concat([df_all, df_cleaned_chosun], ignore_index=True)
df_all = pd.concat([df_all, df_cleaned_genews], ignore_index=True)
df_all = pd.concat([df_all, df_cleaned_jose], ignore_index=True)
df_all = pd.concat([df_all, df_cleaned_industry], ignore_index=True)
df_all = pd.concat([df_all, df_cleaned_segye], ignore_index=True)
df_all = pd.concat([df_all, df_cleaned_sbs], ignore_index=True)
df_all = pd.concat([df_all, df_cleaned_kyeonggi], ignore_index=True)
df_all

,industry,keyword,date,title,content,url
0,건설업,GS건설 신사업,2025.06.19 15:10,"유아이그룹-GS건설, 도심항공교통 사업 상용화 협약",유아이그룹과 GS건설은 지난 17일 서울 종로구에 위치한 GS건설 본사에서 도심항공...,https://www.joongang.co.kr/article/25345012
1,건설업,GS건설 투자 유치,2025.07.01 15:40,"넷폼알앤디, 20억 원 규모 엑스플로인베스트 투자 유치",건축물 유지관리 MRO 테크 스타트업 넷폼알앤디(대표 이승우)가 GS건설의 기업형 ...,https://www.joongang.co.kr/article/25348184
2,건설업,삼성물산 매출,2025.06.10 17:14,"나흘 매출이 3억…동대문 패션부터 ‘3마’까지, 해외에 깃발 꽂는 K패션","지난 4월 24일, 일본 도쿄 시부야의 복합문화공간 ‘미야시타 파크’. ‘마뗑킴’의...",https://www.joongang.co.kr/article/25342690
3,금속제조업,금속제품+철강,2025.06.12 05:00,美관세·中덤핑에 중소 철강사 휘청…제조업 '관절' 꺾인다,“한국산이 300달러(약 41만원)면 중국산은 같은 품질인데도 100달러(약 14만...,https://www.joongang.co.kr/article/25343150
4,금속제조업,비철금속+제련,2025.07.01 11:28,반바지·빙수·장어탕…울산 몰린 대기업 공장들 '쿨대응'보니,요즘 출퇴근 시간 울산 동구 HD현대중공업 울산조선소 정문 앞에 가면 반바지나 샌들...,https://www.joongang.co.kr/article/25348110
...,...,...,...,...,...,...
1201,바이오,삼성바이오로직스+신약개발,2025-06-04 18:32,"'삼성에피스홀딩스', 말로만 신약 개발… ""신약개발 자금 턱없이 부족"" 구호뿐 [한...",\n이 기사는 종합경제매체한양경제기사입니다\n\n\n삼성바이오로직스의 인적분할기업인...,https://www.kyeonggi.com/article/20250604580366
1202,바이오,셀트리온+신약개발,2025-06-10 09:57,"셀트리온, 바이오 USA 참가…신약 파이프라인 확대","\n\n셀트리온이 ‘2025 바이오 인터내셔널 컨벤션(바이오USA)’에 참가, 글로...",https://www.kyeonggi.com/article/20250610580035
1203,반도체제조업,삼성전자 자사주,2025-06-15 22:35,"새 정부 정책 적극 호응, 최대 수혜 기업은? [한양경제]",\n이 기사는 종합경제매체한양경제기사입니다\n\n\n이재명 정부 출범 후 이 대통령...,https://www.kyeonggi.com/article/20250615580173
1204,석유정제,유가+정유사,2025-06-17 18:22,"중동 전쟁의 명암...정유사 줄이고 깎고, 방산주 늘리고 투자하고 [한양경제]",\n이 기사는 종합경제매체한양경제기사입니다\n\n\n이스라엘과 이란의 전쟁이 자칫 ...,https://www.kyeonggi.com/article/20250617580380


In [ ]:
df = df.drop_duplicates(subset=["url"], keep="first")

In [18]:
df_all.to_csv("./google/v4/v4_news.csv", index=False, encoding='utf-8-sig')